# Hansen Ch.17 Panel Data

**Chapter 17 Panel Data**

理论推导与**面向初学者的详细注释**见同目录 `Hansen_Ch17_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：collapsed AB/BB GMM 实证（17.15–17.18）+ 末尾的 **理论结论蒙特卡洛验证**。

> **写给只学过李子奈/陈强的同学：** 面板 = 横截面 + 时间维度，核心是**个体效应** $u_i$。
> - $u_i$ 与 $X$ 相关 ⇒ **固定效应 FE**（减均值消 $u_i$，一致但损失变异）；
> - $u_i$ 与 $X$ 不相关 ⇒ **随机效应 RE**（GLS，高效但需假设）；
> - Hausman 检验决定用 FE 还是 RE；
> - **面板必须聚类 SE**（已验证：不聚类偏小 2.2 倍）；
> - 动态面板（$Y_{i,t-1}$ 作回归元）FE 有 Nickell 偏误 ⇒ Arellano–Bond / Blundell–Bond GMM。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import pinv

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")


def ab_gmm(df, idcol, tcol, ycol, endog_names, exo_names=None, max_ylag=4, step=1, year_fe=True):
    if exo_names is None:
        exo_names = []
    df = df.sort_values([idcol, tcol]).copy()
    all_years = sorted(df[tcol].unique())
    yd_years = all_years[2:] if year_fe else []
    data = []
    for _, g in df.groupby(idcol):
        g = g.sort_values(tcol)
        years = g[tcol].values
        y = g[ycol].values.astype(float)
        End = g[endog_names].values.astype(float) if endog_names else np.zeros((len(g), 0))
        Exo = g[exo_names].values.astype(float) if exo_names else np.zeros((len(g), 0))
        YD = (np.column_stack([(years == yy).astype(float) for yy in yd_years])
              if yd_years else np.zeros((len(g), 0)))
        data.append((years, y, End, Exo, YD))
    k_e, k_x, k_yd = len(endog_names), len(exo_names), len(yd_years)
    k = k_e + k_x + k_yd
    y_lags = list(range(2, max_ylag + 1))
    e_lags = list(range(1, max(max_ylag, 2)))
    n_yiv, n_eiv = len(y_lags), k_e * len(e_lags)
    n_xiv = k_x + k_yd
    n_inst = n_yiv + n_eiv + n_xiv
    ZL, dYL, dXL = [], [], []
    for years, y, End, Exo, YD in data:
        m = len(years)
        parts = ([End] if k_e else []) + ([Exo] if k_x else []) + ([YD] if k_yd else [])
        Xlev = np.column_stack(parts) if parts else np.zeros((m, 0))
        eqsZ, eqsdy, eqsdX = [], [], []
        for j in range(1, m):
            if years[j] - years[j - 1] != 1 or j < 2:
                continue
            dy = y[j] - y[j - 1]
            dX = Xlev[j] - Xlev[j - 1]
            z = np.zeros(n_inst)
            for ii, L in enumerate(y_lags):
                if j - L >= 0:
                    z[ii] = y[j - L]
            pos = n_yiv
            for e in range(k_e):
                for L in e_lags:
                    if j - L >= 0:
                        z[pos] = End[j - L, e]
                    pos += 1
            if n_xiv:
                z[pos:] = dX[k_e:]
            eqsZ.append(z)
            eqsdy.append(dy)
            eqsdX.append(dX)
        if eqsZ:
            ZL.append(np.vstack(eqsZ))
            dYL.append(np.array(eqsdy))
            dXL.append(np.vstack(eqsdX))
    Om1 = np.zeros((n_inst, n_inst))
    ZX = np.zeros((n_inst, k))
    Zy = np.zeros(n_inst)
    for Zi, dyi, dXi in zip(ZL, dYL, dXL):
        ti = len(dyi)
        H = 2 * np.eye(ti)
        for t in range(ti - 1):
            H[t, t + 1] = H[t + 1, t] = -1
        Om1 += Zi.T @ H @ Zi
        ZX += Zi.T @ dXi
        Zy += Zi.T @ dyi
    Om1 = Om1 + 1e-8 * np.trace(Om1) / max(n_inst, 1) * np.eye(n_inst)
    W1 = pinv(Om1)
    theta = pinv(ZX.T @ W1 @ ZX) @ (ZX.T @ W1 @ Zy)
    Om2 = np.zeros((n_inst, n_inst))
    for Zi, dyi, dXi in zip(ZL, dYL, dXL):
        e = dyi - dXi @ theta
        Om2 += np.outer(Zi.T @ e, Zi.T @ e)
    Om2 = Om2 + 1e-10 * np.eye(n_inst)
    if step == 2:
        W = pinv(Om2)
        theta = pinv(ZX.T @ W @ ZX) @ (ZX.T @ W @ Zy)
        Om3 = np.zeros((n_inst, n_inst))
        for Zi, dyi, dXi in zip(ZL, dYL, dXL):
            e = dyi - dXi @ theta
            Om3 += np.outer(Zi.T @ e, Zi.T @ e)
        G = ZX.T @ W @ ZX
        V = pinv(G) @ (ZX.T @ W @ Om3 @ W @ ZX) @ pinv(G)
    else:
        G = ZX.T @ W1 @ ZX
        V = pinv(G) @ (ZX.T @ W1 @ Om2 @ W1 @ ZX) @ pinv(G)
    se = np.sqrt(np.maximum(np.diag(V), 0.0))
    names = list(endog_names) + list(exo_names) + [f"yd{y}" for y in yd_years]
    return dict(theta=theta, se=se, names=names, N=len(ZL), n_inst=n_inst, step=step)


def bb_gmm(df, idcol, tcol, ycol, endog_names, exo_names=None, max_ylag=4, step=1, year_fe=True):
    if exo_names is None:
        exo_names = []
    df = df.sort_values([idcol, tcol]).copy()
    all_years = sorted(df[tcol].unique())
    yd_years = all_years[1:] if year_fe else []
    data = []
    for _, g in df.groupby(idcol):
        g = g.sort_values(tcol)
        years = g[tcol].values
        y = g[ycol].values.astype(float)
        End = g[endog_names].values.astype(float) if endog_names else np.zeros((len(g), 0))
        Exo = g[exo_names].values.astype(float) if exo_names else np.zeros((len(g), 0))
        YD = (np.column_stack([(years == yy).astype(float) for yy in yd_years])
              if yd_years else np.zeros((len(g), 0)))
        data.append((years, y, End, Exo, YD))
    k_e, k_x, k_yd = len(endog_names), len(exo_names), len(yd_years)
    k = k_e + k_x + k_yd
    y_lags = list(range(2, max_ylag + 1))
    e_lags = list(range(1, max(max_ylag, 2)))
    n_yiv, n_eiv = len(y_lags), k_e * len(e_lags)
    n_diff = n_yiv + n_eiv + k_x + k_yd
    n_lev = 1 + k_e
    n_inst = n_diff + n_lev
    ZL, YL, XL = [], [], []
    for years, y, End, Exo, YD in data:
        m = len(years)
        parts = ([End] if k_e else []) + ([Exo] if k_x else []) + ([YD] if k_yd else [])
        Xlev = np.column_stack(parts) if parts else np.zeros((m, 0))
        Zr, Yr, Xr = [], [], []
        for j in range(1, m):
            if years[j] - years[j - 1] != 1 or j < 2:
                continue
            dy = y[j] - y[j - 1]
            dX = Xlev[j] - Xlev[j - 1]
            z = np.zeros(n_inst)
            for ii, L in enumerate(y_lags):
                if j - L >= 0:
                    z[ii] = y[j - L]
            pos = n_yiv
            for e in range(k_e):
                for L in e_lags:
                    if j - L >= 0:
                        z[pos] = End[j - L, e]
                    pos += 1
            z[pos:pos + k_x + k_yd] = dX[k_e:]
            Zr.append(z)
            Yr.append(dy)
            Xr.append(dX)
        for j in range(2, m):
            if years[j] - years[j - 1] != 1 or years[j - 1] - years[j - 2] != 1:
                continue
            z = np.zeros(n_inst)
            z[n_diff] = y[j - 1] - y[j - 2]
            for e in range(k_e):
                z[n_diff + 1 + e] = End[j - 1, e] - End[j - 2, e]
            Zr.append(z)
            Yr.append(y[j])
            Xr.append(Xlev[j])
        if Zr:
            ZL.append(np.vstack(Zr))
            YL.append(np.array(Yr))
            XL.append(np.vstack(Xr))
    Om1 = np.zeros((n_inst, n_inst))
    ZX = np.zeros((n_inst, k))
    Zy = np.zeros(n_inst)
    for Zi, yi, Xi in zip(ZL, YL, XL):
        Om1 += Zi.T @ Zi
        ZX += Zi.T @ Xi
        Zy += Zi.T @ yi
    Om1 = Om1 + 1e-8 * np.trace(Om1) / max(n_inst, 1) * np.eye(n_inst)
    W1 = pinv(Om1)
    theta = pinv(ZX.T @ W1 @ ZX) @ (ZX.T @ W1 @ Zy)
    Om2 = np.zeros((n_inst, n_inst))
    for Zi, yi, Xi in zip(ZL, YL, XL):
        e = yi - Xi @ theta
        Om2 += np.outer(Zi.T @ e, Zi.T @ e)
    Om2 = Om2 + 1e-10 * np.eye(n_inst)
    if step == 2:
        W = pinv(Om2)
        theta = pinv(ZX.T @ W @ ZX) @ (ZX.T @ W @ Zy)
        Om3 = np.zeros((n_inst, n_inst))
        for Zi, yi, Xi in zip(ZL, YL, XL):
            e = yi - Xi @ theta
            Om3 += np.outer(Zi.T @ e, Zi.T @ e)
        G = ZX.T @ W @ ZX
        V = pinv(G) @ (ZX.T @ W @ Om3 @ W @ ZX) @ pinv(G)
    else:
        G = ZX.T @ W1 @ ZX
        V = pinv(G) @ (ZX.T @ W1 @ Om2 @ W1 @ ZX) @ pinv(G)
    se = np.sqrt(np.maximum(np.diag(V), 0.0))
    names = list(endog_names) + list(exo_names) + [f"yd{y}" for y in yd_years]
    return dict(theta=theta, se=se, names=names, N=len(ZL), n_inst=n_inst, step=step)


def show(r, drop_yd=True):
    rows = []
    for n, t, s in zip(r["names"], r["theta"], r["se"]):
        if drop_yd and str(n).startswith("yd"):
            continue
        rows.append({"coef": n, "est": t, "se": s})
    out = pd.DataFrame(rows)
    print(out.to_string(index=False, float_format=lambda x: f"{x:0.4f}"))
    print(f"[N={r['N']}, n_inst={r['n_inst']}, step={r['step']}]")


## 17.15 Capital AR(1)

In [ ]:
ab = pd.read_excel(ROOT / "AB1991/AB1991.xlsx").sort_values(["id", "year"])
ab["k_L1"] = ab.groupby("id")["k"].shift(1)
d = ab.dropna(subset=["k", "k_L1"])
print("Arellano-Bond one-step")
show(ab_gmm(d, "id", "year", "k", ["k_L1"], max_ylag=5, step=1, year_fe=True))
print("Blundell-Bond one-step")
show(bb_gmm(d, "id", "year", "k", ["k_L1"], max_ylag=5, step=1, year_fe=True))


## 17.16 Labor demand

In [ ]:
for c in ["n", "w", "k"]:
    ab[f"{c}_L1"] = ab.groupby("id")[c].shift(1)
d = ab.dropna(subset=["n", "n_L1", "w", "w_L1", "k", "k_L1"])
print("(a) strict exo")
show(ab_gmm(d, "id", "year", "n", endog_names=["n_L1"], exo_names=["w", "w_L1", "k", "k_L1"], max_ylag=5, step=1, year_fe=True))
print("(b) predetermined")
show(ab_gmm(d, "id", "year", "n", endog_names=["n_L1", "w", "w_L1", "k", "k_L1"], max_ylag=5, step=1, year_fe=True))
print("(c) BB system")
show(bb_gmm(d, "id", "year", "n", endog_names=["n_L1", "w", "w_L1", "k", "k_L1"], max_ylag=5, step=1, year_fe=True))


## 17.17–17.18 Invest1993 debt

In [ ]:
inv = pd.read_excel(ROOT / "Invest1993/Invest1993.xlsx").rename(columns={"cusip": "id"})
inv = inv.sort_values(["id", "year"])
for col, new in [("debta", "D"), ("inva", "I"), ("vala", "Q"), ("cfa", "CF")]:
    inv[new] = pd.to_numeric(inv[col], errors="coerce")
    inv[f"{new}_L1"] = inv.groupby("id")[new].shift(1)
d = inv.dropna(subset=["D", "D_L1"])
keep = d.groupby("id").size()
d = d[d.id.isin(keep[keep >= 5].index)]
print("17.17 AB two-step")
show(ab_gmm(d, "id", "year", "D", ["D_L1"], max_ylag=4, step=2, year_fe=True))
print("17.17 BB two-step")
show(bb_gmm(d, "id", "year", "D", ["D_L1"], max_ylag=4, step=2, year_fe=True))
d = inv.dropna(subset=["D", "D_L1", "I_L1", "Q_L1", "CF_L1"])
keep = d.groupby("id").size()
d = d[d.id.isin(keep[keep >= 6].index)]
print("17.18 AB two-step")
show(ab_gmm(d, "id", "year", "D", endog_names=["D_L1", "I_L1", "Q_L1", "CF_L1"], max_ylag=4, step=2, year_fe=True))
print("17.18 BB two-step")
show(bb_gmm(d, "id", "year", "D", endog_names=["D_L1", "I_L1", "Q_L1", "CF_L1"], max_ylag=4, step=2, year_fe=True))


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch17 的核心结论：(1) **FE 一致而混合 OLS 偏**（$u_i$ 与 $X$ 相关时）；(2) **within 变换损失组间变异**（var($\dot X$) ≪ var($X$)）；(3) **$T=2$ 时差分 = FE**（逐样本相同）；(4) **面板必须聚类 SE**（不聚类偏小约 2 倍）。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(17)

# ===== (1) FE 一致 vs 混合 OLS 偏 (u_i 与 X 相关) =====
N, T, beta = 500, 5, 1.0
pooled_est, fe_est = [], []
for r in range(2000):
    u = rng.standard_normal(N) * 2                                # 个体效应
    X = rng.standard_normal((N, T)) + u[:, None] * 0.5            # X 与 u 相关
    eps = rng.standard_normal((N, T))
    Y = beta * X + u[:, None] + eps
    # 混合 OLS（不消 u_i ⇒ 有偏）
    Xf, Yf = X.reshape(-1), Y.reshape(-1)
    b_pool = np.sum(Xf * Yf) / np.sum(Xf ** 2)
    # FE (within: 减个体均值 ⇒ 消 u_i ⇒ 一致)
    Xdot = X - X.mean(1, keepdims=True)
    Ydot = Y - Y.mean(1, keepdims=True)
    b_fe = np.sum(Xdot * Ydot) / np.sum(Xdot ** 2)
    pooled_est.append(b_pool); fe_est.append(b_fe)
print(f"[FE vs Pooled] u_i 与 X 相关 (真 β={beta}):")
print(f"  混合 OLS 均值={np.mean(pooled_est):.4f} (偏!)")
print(f"  FE 均值     ={np.mean(fe_est):.4f} (一致 ✓)")

# ===== (2) Within 变换损失组间变异 =====
X = rng.standard_normal((N, T)) + rng.standard_normal(N)[:, None] * 2
var_X = np.var(X)
var_Xdot = np.var(X - X.mean(1, keepdims=True))
print(f"\n[Within 变换] var(X)={var_X:.3f} > var(X_dot)={var_Xdot:.3f}")
print(f"  损失 {(1-var_Xdot/var_X)*100:.0f}% 变异 ⇒ FE 效率代价")

# ===== (3) T=2: 差分 = FE =====
N, T = 2000, 2
u = rng.standard_normal(N)
X = rng.standard_normal((N, T)) + u[:, None] * 0.5
eps = rng.standard_normal((N, T))
Y = X + u[:, None] + eps
Xdot = X - X.mean(1, keepdims=True); Ydot = Y - Y.mean(1, keepdims=True)
b_fe = np.sum(Xdot * Ydot) / np.sum(Xdot ** 2)
dY = Y[:, 1] - Y[:, 0]; dX = X[:, 1] - X[:, 0]
b_fd = np.sum(dX * dY) / np.sum(dX ** 2)
print(f"\n[T=2] FE={b_fe:.8f}, 差分={b_fd:.8f}, 相同? {np.isclose(b_fe, b_fd)} ✓")

# ===== (4) 面板必须聚类 SE =====
N, T, beta = 200, 10, 1.0
u = rng.standard_normal(N) * 3
X = rng.standard_normal((N, T)) + u[:, None] * 0.3
eps = rng.standard_normal((N, T))
Y = beta * X + u[:, None] + eps
Xf, Yf = X.reshape(-1), Y.reshape(-1)
b = np.sum(Xf * Yf) / np.sum(Xf ** 2); e = Yf - Xf * b
V_nc = np.sum(e ** 2) / (N * T - 2) / np.sum((Xf - Xf.mean()) ** 2)     # 非聚类(错!)
ids = np.repeat(np.arange(N), T)
clu = sum(np.sum(e[ids == i] * Xf[ids == i]) ** 2 for i in range(N))     # 聚类"肉"
V_cl = clu / np.sum((Xf - Xf.mean()) ** 2) ** 2
B = []
for r in range(5000):
    u2 = rng.standard_normal(N) * 3; X2 = rng.standard_normal((N, T)) + u2[:, None] * 0.3
    Y2 = beta * X2 + u2[:, None] + rng.standard_normal((N, T))
    xf, yf = X2.reshape(-1), Y2.reshape(-1)
    B.append(np.sum(xf * yf) / np.sum(xf ** 2))
print(f"\n[聚类 SE] 混合 OLS + 个体效应:")
print(f"  非聚类 SE={np.sqrt(V_nc):.4f} (偏小!)")
print(f"  聚类 SE  ={np.sqrt(V_cl):.4f} (正确)")
print(f"  MC真实   ={np.std(B):.4f} (聚类≈真实)")
print(f"  非聚类偏小约 {np.sqrt(V_cl)/np.sqrt(V_nc):.1f} 倍! ⇒ 面板必须聚类")
